# Export Final Model
**Purpose:** Package the final KMeans (k=5) regimes for the dashboard: a tested prediction function, regime descriptions, and requirements.

**Inputs:** `models/scaler.pkl`, `models/kmeans_k5.pkl`, `data/results/delhi_regimes.csv`, `regime_profiles.csv`

**Outputs:** `models/regime_config.json`, `app/regime_pipeline.py`, `requirements.txt`

## 1. Check saved artifacts

In [1]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd

BASE = Path.cwd().parent
for p in ["models/scaler.pkl", "models/kmeans_k5.pkl", "data/results/delhi_regimes.csv",
          "data/results/regime_profiles.csv", "data/results/kmeans_labels.csv", "data/features/delhi_features.csv"]:
    print(("OK       " if (BASE / p).exists() else "MISSING  ") + p)

scaler = joblib.load(BASE / "models" / "scaler.pkl")
kmeans = joblib.load(BASE / "models" / "kmeans_k5.pkl")
print("scaler features:", list(scaler.feature_names_in_))
print("kmeans features:", list(kmeans.feature_names_in_))
print("clusters:", kmeans.n_clusters)

OK       models/scaler.pkl
OK       models/kmeans_k5.pkl
OK       data/results/delhi_regimes.csv
OK       data/results/regime_profiles.csv
OK       data/results/kmeans_labels.csv
OK       data/features/delhi_features.csv
scaler features: ['PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'SO2', 'O3', 'CO', 'Benzene', 'PM_ratio', 'temp', 'rain', 'wind_speed', 'wind_dir_sin', 'wind_dir_cos']
kmeans features: ['PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'SO2', 'O3', 'CO', 'Benzene', 'PM_ratio', 'temp', 'rain', 'wind_speed']
clusters: 5


## 2. Regime descriptions, colours and profiles (for the dashboard)

In [2]:
import calendar

(BASE / "app").mkdir(exist_ok=True)
(BASE / "models").mkdir(exist_ok=True)

reg = pd.read_csv(BASE / "data" / "results" / "delhi_regimes.csv", parse_dates=["Date"], index_col="Date")
prof = pd.read_csv(BASE / "data" / "results" / "regime_profiles.csv", index_col="regime")

names = {0: "Monsoon washout", 1: "Moderate winter smog", 2: "Pre-monsoon dust",
         3: "Severe stagnant winter smog", 4: "Warm, cleaner days"}

# numbering sanity check: cluster 0 must be the rainiest regime
assert prof.loc["Monsoon washout", "rain"] == prof["rain"].max()

meta = {
    "Monsoon washout": {"color": "#2b7bba", "validation": "strong",
        "description": "Rain-washed air: the cleanest regime, with heavy rain and the lowest particle levels.",
        "evidence": "90% of days with more than 10 mm of rain fall here; 99% of its 2019-2020 days were recovered by a model trained on 2016-2018."},
    "Moderate winter smog": {"color": "#f28e2b", "validation": "good",
        "description": "Cool, dry days with elevated fine particles; air quality is mostly Poor or Very poor.",
        "evidence": "84% of its 2019-2020 days were recovered by a model trained on 2016-2018."},
    "Pre-monsoon dust": {"color": "#b07d4f", "validation": "partial",
        "description": "Hot, dry, windy days where coarse particles dominate (low PM2.5/PM10 ratio); likely dust.",
        "evidence": "Holds 59% to 96% of spring days in 2017-2019 but only 3% in spring 2020; 64% recovered in the time test."},
    "Severe stagnant winter smog": {"color": "#c0392b", "validation": "strong",
        "description": "Cold, calm days with the highest pollution: almost all days are Very poor or Severe.",
        "evidence": "The day after Diwali fell here in all three years; 87% recovered in the time test."},
    "Warm, cleaner days": {"color": "#59a14f", "validation": "partial",
        "description": "Warm days with moderate pollution, cleaner than winter; includes most of the spring 2020 lockdown window.",
        "evidence": "88% of days in the 2020 lockdown window; not recovered as a separate group by the 2016-2018 model."},
}

bins = [0, 50, 100, 200, 300, 400, 1000]
labs = ["Good", "Satisfactory", "Moderate", "Poor", "Very poor", "Severe"]
aqi_mix = pd.crosstab(reg["regime"], pd.cut(reg["AQI"], bins=bins, labels=labs), normalize="index").round(3)

regimes = {}
for name in names.values():
    m = reg.loc[reg["regime"] == name, "month"].value_counts(normalize=True).sort_index()
    regimes[name] = {
        **meta[name],
        "profile": {k: round(float(v), 2) for k, v in prof.loc[name].items()},
        "top_months": [calendar.month_abbr[i] for i in m[m >= 0.10].index],
        "aqi_mix": {k: float(v) for k, v in aqi_mix.loc[name].items()},
    }

config = {
    "names": {str(k): v for k, v in names.items()},
    "order": list(names.values()),
    "model": "KMeans k=5, wind direction excluded",
    "features_used": list(kmeans.feature_names_in_),
    "regimes": regimes,
    "notes": [
        "Regimes are tendencies within a continuum of conditions (silhouette about 0.2), not sharp categories.",
        "Names are interpretations of the profiles; no source-apportionment or fire data was used.",
        "Data: Delhi city average, December 2016 to July 2020; autumn 2016 and June to September 2017 are excluded.",
    ],
}
(BASE / "models" / "regime_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print({n: r["top_months"] for n, r in regimes.items()})

{'Monsoon washout': ['Jul', 'Aug', 'Sep'], 'Moderate winter smog': ['Jan', 'Feb', 'Mar', 'Oct', 'Nov', 'Dec'], 'Pre-monsoon dust': ['Mar', 'Apr', 'May'], 'Severe stagnant winter smog': ['Jan', 'Nov', 'Dec'], 'Warm, cleaner days': ['Mar', 'Apr', 'May', 'Jun']}


## 3. Prediction pipeline (saved as app/regime_pipeline.py)

In [3]:
%%writefile ../app/regime_pipeline.py
"""Regime prediction for the Delhi air-pollution project.

Takes one day of pollutant and weather readings and returns the nearest regime.
"""
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

BASE = Path(__file__).resolve().parent.parent
_scaler = joblib.load(BASE / "models" / "scaler.pkl")
_kmeans = joblib.load(BASE / "models" / "kmeans_k5.pkl")
_config = json.loads((BASE / "models" / "regime_config.json").read_text(encoding="utf-8"))

NAMES = {int(k): v for k, v in _config["names"].items()}
REQUIRED = ["PM2.5", "PM10", "NO", "NO2", "NH3", "SO2", "O3", "CO", "Benzene", "temp", "rain", "wind_speed"]
LOG_COLS = ["PM2.5", "PM10", "NO", "NO2", "NH3", "SO2", "O3", "CO", "Benzene", "rain"]
NON_NEGATIVE = [c for c in REQUIRED if c != "temp"]


def _check(raw):
    missing = [c for c in REQUIRED if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing inputs: {missing}")
    if raw[REQUIRED].isna().any().any():
        raise ValueError("Inputs contain missing values")
    if (raw[NON_NEGATIVE] < 0).any().any():
        raise ValueError("Pollutant, rain and wind inputs must be non-negative")
    if (raw["PM10"] <= 0).any():
        raise ValueError("PM10 must be positive")


def _prepare(raw):
    """Same steps as notebooks 04 and 05: PM ratio, log transform, scaling."""
    _check(raw)
    X = raw[REQUIRED].astype(float).copy()
    X["PM_ratio"] = (X["PM2.5"] / X["PM10"]).clip(upper=1.0)
    X[LOG_COLS] = np.log1p(X[LOG_COLS])
    # wind direction is not used by the final model; neutral values keep the scaler happy
    X["wind_dir_sin"] = 0.0
    X["wind_dir_cos"] = 0.0
    X = X[list(_scaler.feature_names_in_)]
    Z = pd.DataFrame(_scaler.transform(X), index=X.index, columns=X.columns)
    return Z[list(_kmeans.feature_names_in_)]


def predict_regimes(raw):
    """Many days at once. Returns cluster id, regime name and distance to every regime centre."""
    Z = _prepare(raw)
    dist = _kmeans.transform(Z)
    labels = _kmeans.predict(Z)
    out = pd.DataFrame({"cluster": labels, "regime": [NAMES[i] for i in labels]}, index=raw.index)
    for i in range(dist.shape[1]):
        out[f"dist_{NAMES[i]}"] = dist[:, i]
    return out


def predict_regime(**inputs):
    """One day. Distances are not probabilities; a small margin means a borderline day."""
    res = predict_regimes(pd.DataFrame([inputs])).iloc[0]
    dist = {NAMES[i]: float(res[f"dist_{NAMES[i]}"]) for i in range(len(NAMES))}
    ranked = sorted(dist.items(), key=lambda kv: kv[1])
    (first, d1), (second, d2) = ranked[0], ranked[1]
    return {
        "regime": first,
        "runner_up": second,
        "margin": round((d2 - d1) / d2, 3),
        "distances": {k: round(v, 2) for k, v in ranked},
    }

Writing ../app/regime_pipeline.py


## 4. Round-trip test: raw features in, saved labels out

In [4]:
import sys, importlib
sys.path.insert(0, str(BASE / "app"))
import regime_pipeline as rp
importlib.reload(rp)

raw = pd.read_csv(BASE / "data" / "features" / "delhi_features.csv", parse_dates=["Date"], index_col="Date")
saved = pd.read_csv(BASE / "data" / "results" / "kmeans_labels.csv", parse_dates=["Date"], index_col="Date")["kmeans_cluster"]

pred = rp.predict_regimes(raw[rp.REQUIRED])
same = (pred["cluster"] == saved)
print("days reproduced exactly:", int(same.sum()), "of", len(saved), f"({same.mean():.1%})")

days reproduced exactly: 1195 of 1195 (100.0%)


## 5. Examples: median day of each regime and two custom days

In [5]:
med = raw[rp.REQUIRED].groupby(saved.map(rp.NAMES)).median()
for name, row in med.iterrows():
    r = rp.predict_regime(**row.to_dict())
    print(f"{name:30s} -> {r['regime']:30s} margin {r['margin']}")

print(rp.predict_regime(**{"PM2.5": 310, "PM10": 430, "NO": 90, "NO2": 75, "NH3": 55, "SO2": 18,
                           "O3": 40, "CO": 2.2, "Benzene": 7.5, "temp": 15, "rain": 0, "wind_speed": 9}))
print(rp.predict_regime(**{"PM2.5": 35, "PM10": 80, "NO": 12, "NO2": 25, "NH3": 30, "SO2": 9,
                           "O3": 30, "CO": 0.9, "Benzene": 1.7, "temp": 28, "rain": 12, "wind_speed": 16}))

Moderate winter smog           -> Moderate winter smog           margin 0.873
Monsoon washout                -> Monsoon washout                margin 0.897
Pre-monsoon dust               -> Pre-monsoon dust               margin 0.867
Severe stagnant winter smog    -> Severe stagnant winter smog    margin 0.91
Warm, cleaner days             -> Warm, cleaner days             margin 0.898
{'regime': 'Severe stagnant winter smog', 'runner_up': 'Moderate winter smog', 'margin': 0.696, 'distances': {'Severe stagnant winter smog': 1.17, 'Moderate winter smog': 3.86, 'Pre-monsoon dust': 4.98, 'Warm, cleaner days': 6.94, 'Monsoon washout': 7.77}}
{'regime': 'Monsoon washout', 'runner_up': 'Warm, cleaner days', 'margin': 0.703, 'distances': {'Monsoon washout': 1.12, 'Warm, cleaner days': 3.75, 'Moderate winter smog': 5.26, 'Pre-monsoon dust': 5.9, 'Severe stagnant winter smog': 7.7}}


## 6. Input validation

In [6]:
try:
    rp.predict_regime(**{"PM2.5": -5, "PM10": 80, "NO": 12, "NO2": 25, "NH3": 30, "SO2": 9,
                         "O3": 30, "CO": 0.9, "Benzene": 1.7, "temp": 28, "rain": 0, "wind_speed": 16})
except ValueError as e:
    print("rejected as expected:", e)

rejected as expected: Pollutant, rain and wind inputs must be non-negative


## 7. Requirements file

In [7]:
from importlib.metadata import version, PackageNotFoundError
pkgs = ["pandas", "numpy", "scikit-learn", "joblib", "scipy", "matplotlib", "seaborn", "streamlit", "plotly"]
lines = []
for p in pkgs:
    try:
        lines.append(f"{p}=={version(p)}")
    except PackageNotFoundError:
        print("not installed yet (add it when the app needs it):", p)
(BASE / "requirements.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

pandas==3.0.3
numpy==2.5.1
scikit-learn==1.9.1
joblib==1.6.0
scipy==1.18.1
matplotlib==3.11.1
seaborn==0.13.2
streamlit==1.63.0
plotly==7.1.0


## 8. Project contents

In [8]:
for folder in ["data/raw", "data/cleaned", "data/features", "data/results", "models", "app", "notebooks"]:
    print(folder, "->", sorted(p.name for p in (BASE / folder).glob("*")))

data/raw -> ['city_day.csv', 'weather_delhi.csv']
data/cleaned -> ['.ipynb_checkpoints', 'delhi_clean.csv', 'delhi_merged.csv']
data/features -> ['delhi_context.csv', 'delhi_features.csv', 'delhi_pca.csv', 'delhi_scaled.csv']
data/results -> ['delhi_regimes.csv', 'gmm_labels.csv', 'gmm_probabilities.csv', 'hdbscan_mcs10_ms5_labels.csv', 'hdbscan_mcs20_ms10_labels.csv', 'kmeans_labels.csv', 'method_agreement.csv', 'method_comparison.csv', 'regime_profiles.csv']
models -> ['gmm_k5_noSO2.pkl', 'kmeans_k5.pkl', 'pca6.pkl', 'regime_config.json', 'scaler.pkl']
app -> ['__pycache__', 'regime_pipeline.py']
notebooks -> ['.ipynb_checkpoints', '01_data_collection.ipynb.ipynb', '02_eda.ipynb', '03_data_cleaning.ipynb', '04_feature_engineering.ipynb', '05_scaling_and_pca.ipynb', '06_kmeans.ipynb', '07_hdbscan.ipynb', '08_gaussian_mixture.ipynb', '09_algorithm_comparison.ipynb', '10_regime_profiling.ipynb', '11_validation.ipynb', '12_export_final_model.ipynb']


## Conclusion
The saved pipeline reproduces all 1,195 labelled days (100%) from raw readings.
The median day of each regime maps back to its own regime (margins 0.87 to 0.91), and two custom test days (a cold, calm, polluted day and a rainy July day) land in the severe winter smog and monsoon regimes as expected.
It accepts 12 inputs (9 pollutants plus temperature, rain and wind speed), validates them, and returns the nearest regime with distances to all regimes.
Distances are not probabilities. Regimes overlap, so a small margin between the nearest and second-nearest regime marks a borderline day.